In [ ]:
# -*- coding: utf-8 -*-
# Size (MB, FP32, log-x) vs MPJPE (mm): two datasets (mmBody & MMFi)

import os
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from cycler import cycler

# ------------------------------
# Style Configuration
# ------------------------------
def get_transparent_color(color, transparency=0.5):
    """Convert a hex color to a lighter (more transparent) version."""
    c = mcolors.hex2color(color)
    c = [c[0] * transparency + (1.0 - transparency),
         c[1] * transparency + (1.0 - transparency),
         c[2] * transparency + (1.0 - transparency)]
    return "#{:02X}{:02X}{:02X}".format(int(c[0]*255), int(c[1]*255), int(c[2]*255))

palette = ['#1e90ff', '#ffbb00', '#ff5080', '#a7426d', '#ff3c10', "#282828"]  # Bocchi palette
markers = ['o', 'P', '^', 's', 'p', 'h']  # 6 filled markers

plt.rcdefaults()
plt.rcParams["figure.figsize"] = [8, 4]
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300

plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["ytick.direction"] = "in"

plt.rcParams['lines.linewidth'] = 2.5
plt.rcParams['lines.markersize'] = 8
plt.rcParams['lines.markeredgewidth'] = 2.0

plt.rcParams["font.size"] = 18
plt.rcParams["font.family"] = "Arial"
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

plt.rcParams["legend.fontsize"] = "medium"
plt.rcParams["legend.facecolor"] = "white"
plt.rcParams["legend.edgecolor"] = "white"
plt.rcParams["legend.framealpha"] = 0.9
plt.rcParams['legend.frameon'] = False
plt.rcParams['legend.handlelength'] = 1.5
plt.rcParams['legend.handletextpad'] = 0.5
plt.rcParams['legend.columnspacing'] = 0.8
plt.rcParams['legend.labelspacing'] = 0.3

plt.rcParams["axes.prop_cycle"] = cycler(color=palette) + cycler(marker=markers)


In [ ]:
# ------------------------------
# Data Loading
# ------------------------------

# Model sizes in MB (FP32)
size_mb = {
    "xlarge": 592.85,
    "large": 283.54,
    "base": 102.64,
    "small": 60.00,
    "tiny": 20.25,
    "micro": 6.00,
    "nano": 3.38,
    # "pico": 1.50,
}

# Size name mapping for different naming conventions in checkpoint folders
size_name_map = {
    "xlarge": ["xlarge", "xl"],
    "large": ["large"],
    "base": ["base"],
    "small": ["small"],
    "tiny": ["tiny"],
    "micro": ["micro"],
    "nano": ["nano"],
}

def load_mpjpe_from_checkpoints(dataset_name: str, base_path: str = "../../../logs/pose_estimation") -> dict:
    """
    Automatically load test MPJPE from checkpoint best_metrics.json files.
    
    Args:
        dataset_name: 'mmbody' or 'mmfi'
        base_path: path to logs/pose_estimation directory
    
    Returns:
        dict mapping size names to MPJPE values (0 if not found)
    """
    checkpoint_dir = os.path.join(base_path, dataset_name, "checkpoints")
    mpjpe_dict = {}
    
    for size_name, patterns in size_name_map.items():
        mpjpe_dict[size_name] = 0  # Default to 0 if not found
        
        for pattern in patterns:
            # Find checkpoint folder containing this size pattern
            # Pattern: PointTransformer-{dataset}_{size}_job*
            search_pattern = os.path.join(checkpoint_dir, f"*_{pattern}_*", "best_metrics.json")
            matches = glob.glob(search_pattern)
            
            if matches:
                # Use the first match (or latest if multiple)
                metrics_file = matches[0]
                try:
                    with open(metrics_file, 'r') as f:
                        metrics = json.load(f)
                        mpjpe_dict[size_name] = metrics.get("test", {}).get("mpjpe", 0)
                        print(f"[{dataset_name}] {size_name}: {mpjpe_dict[size_name]:.2f} mm (from {os.path.basename(os.path.dirname(metrics_file))})")
                        break
                except Exception as e:
                    print(f"[{dataset_name}] {size_name}: Error reading {metrics_file}: {e}")
        
        if mpjpe_dict[size_name] == 0:
            print(f"[{dataset_name}] {size_name}: Not found (using 0)")
    
    return mpjpe_dict

def build_df(mpjpe_dict, size_map):
    """Build a DataFrame from MPJPE dict, filtering out zero values and sorting by size."""
    # Only keep scales that exist in both dicts and have non-zero MPJPE; sort by size ascending
    keys = [k for k in mpjpe_dict.keys() if k in size_map and mpjpe_dict[k] > 0]
    keys = sorted(keys, key=lambda k: size_map[k])
    return pd.DataFrame({
        "scale": keys,
        "size_mb": [size_map[k] for k in keys],
        "mpjpe_mm": [mpjpe_dict[k] for k in keys],
    })

# Load MPJPE values automatically from checkpoint files
print("Loading mmBody checkpoints...")
mpjpe_mmbody = load_mpjpe_from_checkpoints("mmbody")
print("\nLoading MMFi checkpoints...")
mpjpe_mmfi = load_mpjpe_from_checkpoints("mmfi")

# Build DataFrames for plotting
df_mmbody = build_df(mpjpe_mmbody, size_mb)
df_mmfi = build_df(mpjpe_mmfi, size_mb)

print("\n" + "="*50)
print("mmBody DataFrame:")
print(df_mmbody)
print("\nMMFi DataFrame:")
print(df_mmfi)


In [ ]:
# ------------------------------
# Plotting
# ------------------------------
fig, ax = plt.subplots()

# Use specific colors and markers to avoid confusion with rc cycle
line1, = ax.plot(
    df_mmbody["size_mb"], df_mmbody["mpjpe_mm"],
    label="mmBody", color=palette[0], marker='o', zorder=3
)
line2, = ax.plot(
    df_mmfi["size_mb"], df_mmfi["mpjpe_mm"],
    label="MMFi", color=palette[1], marker='s', zorder=3
)

# Annotate data points with MPJPE values (offset vertically to reduce overlap)
for _, r in df_mmbody.iterrows():
    ax.text(r["size_mb"], r["mpjpe_mm"] + 1, f"{r['mpjpe_mm']:.2f}",
            ha='center', va='bottom', fontsize=16)
for _, r in df_mmfi.iterrows():
    ax.text(r["size_mb"], r["mpjpe_mm"] + 1, f"{r['mpjpe_mm']:.2f}",
            ha='center', va='bottom', fontsize=16)

ax.set_xscale("log")
ax.set_xlabel("Model size (MB, FP32)")
ax.set_ylabel("MPJPE (mm)")

# Set Y-axis range with margin
ax.set_ylim(65, 95)

# Set X-axis range
ax.set_xlim(left=1, right=1000)

ax.grid(True, which="both", axis="both", linestyle="--", alpha=0.5)

# Format tick labels
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, pos: f"{v:g}"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, pos: f"{v:g}"))

# Legend at top center
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.2), ncol=2, frameon=False, fontsize=18)
plt.tight_layout()

# ------------------------------
# Save Figure
# ------------------------------
plt.savefig("size_vs_mpjpe_dual.pdf", bbox_inches="tight")
plt.show()
